In [20]:
########## 1. Installing and Importing libraries
import pandas as pd
import numpy as np
import geopandas as gpd
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.linear_model import LassoCV
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from pykrige.rk import RegressionKriging
# Scaling the input features
from sklearn.preprocessing import StandardScaler
# To prevent numpy floats to be displayed as np.float64()
np.set_printoptions(legacy="1.21")
# To ignore warning messages
import warnings
warnings.filterwarnings("ignore")

In [3]:
################ 2. Loading the dataset
data = pd.read_csv('E:/VData/SData/Wells4.csv')

In [4]:
# Random seed for reproducibility purposes
np.random.seed(42)

In [6]:
data.head()

,x_coord,y_coord,Depth,pH,SO4,NO3,T
0,17.74,15.57,259.0,7.00,39.0,9.0,27.0
1,17.48,15.82,220.0,6.60,17.0,12.0,34.7
2,17.48,15.83,220.0,6.00,925.0,5.0,27.0
3,17.39,15.37,286.0,7.00,3300.0,1.0,29.0
4,17.35,13.80,47.0,7.45,13.0,0.2,26.5


In [6]:
############# 3. Data Preparation

In [7]:
# a. Target variable, y
y = data['pH']

In [8]:
y.shape

(1311,)

In [9]:
# b. Feature variables or predictors, p
p = []
p.append(data['T'])
p.append(data['Depth'])
p = np.array(p).T

In [10]:
p.shape

(1311, 2)

In [22]:
print(p[:5])

[[ 27.  259. ]
 [ 34.7 220. ]
 [ 27.  220. ]
 [ 29.  286. ]
 [ 26.5  47. ]]


In [11]:
# Scaling features
# Initializing and fitting the scaler
scaler = StandardScaler()
scaler.fit(p)
StandardScaler()
# Scaling P
p = scaler.transform(p)

In [12]:
p.shape

(1311, 2)

In [13]:
print('The scaled ps are:', p[:5])

The scaled ps are: [[-0.51360148 10.96104802]
 [ 2.2104606   8.92181417]
 [-0.51360148  8.92181417]
 [ 0.19394711 12.37282529]
 [-0.69048863 -0.12401799]]


In [14]:
# c. The Coordinates: Lon/Lat points
X =[]
X.append(data['x_coord'])
X.append(data['y_coord'])
X = np.array(X).T

In [15]:
X.shape

(1311, 2)

In [16]:
print(X[:5])

[[17.74 15.57]
 [17.48 15.82]
 [17.48 15.83]
 [17.39 15.37]
 [17.35 13.8 ]]


In [17]:
## d. Splitting the dataset into training and testing data
p_train, p_test, X_train, X_test, y_train, y_test = train_test_split(p, X, y, test_size = 0.3, random_state = 42)

In [21]:
########### 4. Setting up machine learning model
lr_model = LinearRegression( copy_X = True, fit_intercept = False)
rg_model = Ridge(alpha = 100, fit_intercept = True, solver ='svd')
ls_model = Lasso(alpha = 0.0001, selection = 'cyclic', max_iter = 2000)
el_model = ElasticNet(alpha = 0.0001, l1_ratio = 0.23, selection = 'random', max_iter = 3000)
svr_model = SVR(C = 120, epsilon = 0.01, kernel = 'linear')
rf_model = RandomForestRegressor(max_depth = 3, max_leaf_nodes = 2, min_samples_split = 6, n_estimators = 50)
rt_model = DecisionTreeRegressor(max_depth = 3, max_features = 1, max_leaf_nodes = 50, min_samples_split = 2, min_samples_leaf = 5)
xtr_model = ExtraTreesRegressor(max_depth = 3, max_features = 1, max_leaf_nodes = 100, min_samples_leaf = 4, min_samples_split = 10, n_estimators = 50)
gbr_model = GradientBoostingRegressor(learning_rate = 0.01, max_depth = 4, n_estimators = 100, subsample = 0.5)
xgb_model = XGBRegressor(colsample_bytree = 0.1, learning_rate = 0.001, max_depth = 9, n_estimators = 500, subsample = 0.3)

In [22]:
############# 5.Nesting the ML models, training and Fitting the regression kriging, and printing the metrics
models = [lr_model, rg_model, ls_model, el_model, svr_model, rf_model, rt_model, xtr_model, gbr_model, xgb_model]
for m in models:
    print("=" * 40)
    print("regression model:", m.__class__.__name__)
    m_rk = RegressionKriging(regression_model = m, n_closest_points = 10)
    m_rk.fit(p_train, X_train, y_train)
    print("Regression Score: ", m_rk.regression_model.score(p_test, y_test))
    print("RK score: ", m_rk.score(p_test, X_test, y_test))


regression model: LinearRegression
Finished learning regression model
Finished kriging residuals
Regression Score:  -153.67132485548532
RK score:  -0.023174494660606504
regression model: Ridge
Finished learning regression model
Finished kriging residuals
Regression Score:  -0.007508386668742428
RK score:  0.044403805779482775
regression model: Lasso
Finished learning regression model
Finished kriging residuals
Regression Score:  -0.0094033974468537
RK score:  0.044210731665579095
regression model: ElasticNet
Finished learning regression model
Finished kriging residuals
Regression Score:  -0.009415793384527582
RK score:  0.044201732165018104
regression model: SVR
Finished learning regression model
Finished kriging residuals
Regression Score:  0.0009007423137650239
RK score:  0.042206895932618615
regression model: RandomForestRegressor
Finished learning regression model
Finished kriging residuals
Regression Score:  0.0003962736676420553
RK score:  0.07178216223409739
regression model: De

In [ ]:
#######################################################################